In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [2]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake["label"] = "Fake"
true["label"] = "Real"

df = pd.concat([fake, true], ignore_index=True)

# Fill missing text if needed
fake["text"] = fake["text"].fillna("")
true["text"] = true["text"].fillna("")
df["text"] = df["text"].fillna("")

In [3]:
def print_topics(model, feature_names, n_top_words=10, title="Topics"):
    print(f"\n{title}")
    print("-" * 50)
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[-n_top_words:][::-1]]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

In [4]:
def run_lda(text_data, n_topics=5, max_df=0.95, min_df=10):
    vectorizer = CountVectorizer(
        stop_words="english",
        max_df=max_df,
        min_df=min_df
    )
    
    X = vectorizer.fit_transform(text_data)
    
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42
    )
    
    lda.fit(X)
    
    return lda, vectorizer, X

In [5]:
fake_lda, fake_vectorizer, X_fake = run_lda(fake["text"], n_topics=5)

print_topics(
    fake_lda,
    fake_vectorizer.get_feature_names_out(),
    n_top_words=10,
    title="Fake News Topics"
)


Fake News Topics
--------------------------------------------------
Topic 1: media, government, war, american, new, world, state, syria, obama, military
Topic 2: said, police, black, people, video, just, man, year, told, gun
Topic 3: said, people, state, federal, obama, court, government, states, law, 000
Topic 4: trump, president, donald, people, just, said, like, obama, image, republican
Topic 5: clinton, trump, hillary, news, said, president, fbi, campaign, russia, russian


In [6]:
fake_topic_dist = fake_lda.transform(X_fake)
fake["dominant_topic"] = fake_topic_dist.argmax(axis=1) + 1

fake["dominant_topic"].value_counts().sort_index()

dominant_topic
1    2850
2    4664
3    3712
4    9058
5    3197
Name: count, dtype: int64

In [7]:
real_lda, real_vectorizer, X_real = run_lda(true["text"], n_topics=5)

print_topics(
    real_lda,
    real_vectorizer.get_feature_names_out(),
    n_top_words=10,
    title="Real News Topics"
)


Real News Topics
--------------------------------------------------
Topic 1: said, north, united, korea, state, military, states, nuclear, russia, trump
Topic 2: said, court, people, state, police, law, government, rights, states, case
Topic 3: said, percent, tax, year, government, party, election, people, million, new
Topic 4: said, china, minister, president, eu, party, european, government, prime, britain
Topic 5: trump, said, republican, house, president, clinton, senate, campaign, washington, white


In [8]:
real_topic_dist = real_lda.transform(X_real)
true["dominant_topic"] = real_topic_dist.argmax(axis=1) + 1

true["dominant_topic"].value_counts().sort_index()

dominant_topic
1    4176
2    3710
3    3547
4    3442
5    6542
Name: count, dtype: int64

In [9]:
mix_lda, mix_vectorizer, X_mix = run_lda(df["text"], n_topics=5)

print_topics(
    mix_lda,
    mix_vectorizer.get_feature_names_out(),
    n_top_words=10,
    title="Mixed Fake + Real News Topics"
)


Mixed Fake + Real News Topics
--------------------------------------------------
Topic 1: said, reuters, united, president, government, north, minister, state, states, china
Topic 2: said, people, police, trump, black, just, like, right, gun, white
Topic 3: said, state, government, court, year, reuters, states, new, federal, people
Topic 4: trump, president, just, like, people, donald, obama, hillary, twitter, clinton
Topic 5: trump, said, president, republican, clinton, house, campaign, election, donald, senate


In [10]:
mix_topic_dist = mix_lda.transform(X_mix)
df["dominant_topic"] = mix_topic_dist.argmax(axis=1) + 1

df["dominant_topic"].value_counts().sort_index()

dominant_topic
1    10449
2     6881
3     8365
4    10655
5     8548
Name: count, dtype: int64

In [11]:
pd.crosstab(df["dominant_topic"], df["label"])

label,Fake,Real
dominant_topic,,
1,1516,8933
2,6072,809
3,2098,6267
4,10279,376
5,3516,5032
